## Analysing SegFormer segmentation model
---

In this notebook, we are going to fine-tune SegFormerForSemanticSegmentation on a custom semantic segmentation dataset. In semantic segmentation, the goal for the model is to label each pixel of an image with one of a list of predefined classes.

## Imports 
---

In [ ]:
#external
import pandas as pd

#model
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

#metrics
import evaluate
from torch import nn

#utils
from src.utils.dataset import load_foodseg103, decode_image_from_bytes
from src.utils.visualization import predict_random_images

#constants
from src.constants.category_id import CATEGORY_ID

## Testing on custom dataset
---

In [ ]:
SEED = 42
IMAGE_SIZE = 64
LEARNING_RATE = 2e-05
BATCH_SIZE = 8
NUM_EPOCHS = 10

### Defining Dataset

In [ ]:
class SemanticSegmentationFoodDataset(Dataset):
    def __init__(self, image_dataset_df:pd.DataFrame, image_processor:SegformerImageProcessor):
        self.image_dataset = image_dataset_df.reset_index(drop=True)
        self.image_processor = image_processor

    def __len__(self):
        return self.image_dataset.shape[0]
    
    def __getitem__(self, index):
        image_information = self.image_dataset.loc[index]
        image_decoded = decode_image_from_bytes(image_information["image"])
        mask = decode_image_from_bytes(image_information["label"])
        encoded_inputs = self.image_processor(image_decoded, mask, return_tensors="pt")
        for k,v in encoded_inputs.items():
          encoded_inputs[k].squeeze_() # remove batch dimension
        return encoded_inputs

In [ ]:
df = load_foodseg103(type="all")
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED)
test_df, val_df = train_test_split(temp_df, test_size=0.3, random_state=SEED)

In [ ]:
image_processor = SegformerImageProcessor(
    do_reduce_labels=False,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE}
)

In [ ]:
train_dataset = SemanticSegmentationFoodDataset(train_df, image_processor)
val_dataset = SemanticSegmentationFoodDataset(val_df, image_processor)
test_dataset = SemanticSegmentationFoodDataset(test_df, image_processor)

In [ ]:
print(f"Number of training examples: {train_dataset.__len__()}")
print(f"Number of validation examples: {val_dataset.__len__()}")
print(f"Number of testing examples: {test_dataset.__len__()}")

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, prefetch_factor=2)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, prefetch_factor=2)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=2, prefetch_factor=2)

### Model definition

In [ ]:
class SegformerFinetuner(pl.LightningModule):
    def __init__(self, id2label, train_dataloader=None, val_dataloader=None, test_dataloader=None, metrics_interval=100):
        super(SegformerFinetuner, self).__init__()
        self.id2label = id2label
        self.metrics_interval = metrics_interval
        self.train_dl = train_dataloader
        self.val_dl = val_dataloader
        self.test_dl = test_dataloader
        
        self.num_classes = len(id2label.keys())
        self.label2id = {v:k for k,v in self.id2label.items()}
        
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0", 
            return_dict=False, 
            num_labels=self.num_classes,
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True,
        )
        for param in self.model.base_model.parameters():
            param.requires_grad = False
            
        self.train_mean_iou = evaluate.load("mean_iou")
        self.val_mean_iou = evaluate.load("mean_iou")
        self.test_mean_iou = evaluate.load("mean_iou")

        self.validation_step_outputs = []
        self.test_step_outputs = []
        
    def forward(self, images, masks):
        outputs = self.model(pixel_values=images, labels=masks)
        return(outputs)
    
    def training_step(self, batch, batch_nb):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs[0], outputs[1]
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=masks.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.train_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(), 
            references=masks.detach().cpu().numpy()
        )
        if batch_nb % self.metrics_interval == 0:
            metrics = self.train_mean_iou.compute(
                num_labels=self.num_classes, 
                ignore_index=255, 
                reduce_labels=False,
            )
            metrics = {'loss': loss, "mean_iou": metrics["mean_iou"], "mean_accuracy": metrics["mean_accuracy"]}
            for k,v in metrics.items():
                self.log(k,v)
            return(metrics)
        else:
            return({'loss': loss})
    
    def validation_step(self, batch, batch_nb):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs[0], outputs[1]
        self.validation_step_outputs.append(loss)
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=masks.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.val_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(), 
            references=masks.detach().cpu().numpy()
        )
        return({'val_loss': loss})
    
    def on_validation_epoch_end(self):
        metrics = self.val_mean_iou.compute(
              num_labels=self.num_classes, 
              ignore_index=255, 
              reduce_labels=False,
          )
        avg_val_loss = torch.stack(self.validation_step_outputs).mean()
        val_mean_iou = metrics["mean_iou"]
        val_mean_accuracy = metrics["mean_accuracy"]
        metrics = {"val_loss": avg_val_loss, "val_mean_iou":val_mean_iou, "val_mean_accuracy":val_mean_accuracy}
        for k,v in metrics.items():
            self.log(k,v)
        self.validation_step_outputs.clear()
        return metrics
    
    def test_step(self, batch, batch_nb):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs[0], outputs[1]
        self.test_step_outputs.append(loss)
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=masks.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.test_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(), 
            references=masks.detach().cpu().numpy()
        )
        return({'test_loss': loss})
    
    def on_test_epoch_end(self):
        metrics = self.test_mean_iou.compute(
              num_labels=self.num_classes, 
              ignore_index=255, 
              reduce_labels=False,
          )
        avg_test_loss = torch.stack(self.test_step_outputs).mean()
        test_mean_iou = metrics["mean_iou"]
        test_mean_accuracy = metrics["mean_accuracy"]
        metrics = {"test_loss": avg_test_loss, "test_mean_iou":test_mean_iou, "test_mean_accuracy":test_mean_accuracy}
        for k,v in metrics.items():
            self.log(k,v)
        self.test_step_outputs.clear()
        return metrics
    
    def configure_optimizers(self):
        return torch.optim.Adam([p for p in self.parameters() if p.requires_grad], lr=LEARNING_RATE)
    
    def train_dataloader(self):
        return self.train_dl
    
    def val_dataloader(self):
        return self.val_dl
    
    def test_dataloader(self):
        return self.test_dl

In [ ]:
segformer_finetuner = SegformerFinetuner(
    CATEGORY_ID, 
    train_dataloader=train_dataloader, 
    val_dataloader=val_dataloader, 
    test_dataloader=test_dataloader, 
    metrics_interval=10
)

In [ ]:
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint

early_stop_callback = EarlyStopping(
    monitor="val_loss", 
    min_delta=0.00, 
    patience=10, 
    verbose=False, 
    mode="min",
)

checkpoint_callback = ModelCheckpoint(save_top_k=1, monitor="val_loss")

trainer = pl.Trainer(
    callbacks=[early_stop_callback, checkpoint_callback],
    max_epochs=NUM_EPOCHS,
    val_check_interval=len(train_dataloader)
)

In [ ]:
trainer.fit(segformer_finetuner)

In [ ]:
res = trainer.test(ckpt_path="best")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/

## References
[1] https://github.com/NielsRogge/Transformers-Tutorials/tree/master/SegFormer
[2] https://blog.roboflow.com/how-to-train-segformer-on-a-custom-dataset-with-pytorch-lightning/#create-a-dataset